In [17]:
import geopandas as gpd
from shapely.validation import make_valid
import pandas as pd

# 1. 读取数据并转换到投影坐标系
target_crs = 'EPSG:32647' 
gdf_gt = gpd.read_file("2005_truth.gpkg")
gdf_pred = gpd.read_file("2005_pred_masked.gpkg")

# 转换坐标系
gdf_gt = gdf_gt.to_crs(target_crs)
gdf_pred = gdf_pred.to_crs(target_crs)


# 2. 检查并修复无效几何（包括空值处理）
def fix_geometries(gdf):
    """修复 GeoDataFrame 中的所有无效几何，并处理空值"""
    # 先检查是否有空几何（None）--raster_vector 转换过程中可能会产生空值
    null_mask = gdf.geometry.isna()
    null_count = null_mask.sum()
    if null_count > 0:
        print(f"  发现 {null_count} 个空几何，将被删除。")
        gdf = gdf.loc[~null_mask].copy()
    
    # 再检查无效几何（非空但拓扑无效）
    invalid_mask = ~gdf.is_valid
    invalid_count = invalid_mask.sum()
    if invalid_count > 0:
        print(f"  发现 {invalid_count} 个无效几何，正在修复...")
        # 仅对无效几何应用 make_valid，保持有效几何不变
        gdf.loc[invalid_mask, 'geometry'] = gdf.loc[invalid_mask, 'geometry'].apply(
            lambda geom: make_valid(geom)
        )
    else:
        print("  所有几何均有效，无需修复。")
    return gdf

print("\n检查真实数据几何有效性:")
gdf_gt = fix_geometries(gdf_gt)
print("检查预测数据几何有效性：")
gdf_pred = fix_geometries(gdf_pred)

# 3. 合并所有多边形为单一几何体
gt_union = gdf_gt.geometry.union_all()
pred_union = gdf_pred.geometry.union_all()

# 4. 计算 IoU
intersection = gt_union.intersection(pred_union).area
union = gt_union.union(pred_union).area
iou = intersection / union

print(f"真实水体总面积  : {gt_union.area / 1e6:.4f} km²")
print(f"预测水体总面积  : {pred_union.area / 1e6:.4f} km²")
print(f"交集面积        : {intersection / 1e6:.4f} km²")
print(f"并集面积        : {union / 1e6:.4f} km²")
print(f"水体 IoU         : {iou:.6f}")


检查真实数据几何有效性:
  发现 205 个空几何，将被删除。
  发现 1 个无效几何，正在修复...
检查预测数据几何有效性：
  所有几何均有效，无需修复。
真实水体总面积  : 4314.1832 km²
预测水体总面积  : 4294.0379 km²
交集面积        : 4285.5189 km²
并集面积        : 4322.7022 km²
水体 IoU         : 0.991398
